# 00 - Database & SQL
CareFlow | Yash Rahangdale | Infotact Solutions

Yahan hum event log ko SQL (SQLite demo / MySQL / PostgreSQL) me load karenge aur KPI queries chalayenge.

In [1]:
import sys, pandas as pd
sys.path.append('../src')
from connect_db import get_engine, build_sqlite, run_kpis
eng = get_engine('sqlite')
build_sqlite(eng)

[connect_db] engine = sqlite
[connect_db] loaded 1,500 cases / 13,664 events


## Overall KPI

In [2]:
pd.read_sql('SELECT COUNT(*) cases, ROUND(AVG(los_hours),2) avg_los_h, ROUND(SUM(total_cost_inr),0) total_cost FROM fact_case', eng)

,cases,avg_los_h,total_cost
0,1500,9.65,25886761.0


## Bottleneck: activity-wise waiting time (pure SQL)

In [3]:
pd.read_sql('''SELECT activity, COUNT(*) occurrences, ROUND(AVG(wait_minutes),1) avg_wait_min,
 ROUND(AVG(service_minutes),1) avg_service_min FROM fact_event GROUP BY activity ORDER BY avg_wait_min DESC''', eng)

,activity,occurrences,avg_wait_min,avg_service_min
0,Surgery,386,83.0,202.9
1,Observation,583,69.4,153.7
2,Radiology,680,54.1,44.7
3,ICU Transfer,228,54.0,70.5
4,Lab Test,1690,46.5,49.7
5,Admission,379,29.4,40.6
6,Discharge Planning,989,25.9,35.2
7,Specialist Consultation,928,23.4,30.5
8,Physiotherapy,862,21.6,48.7
9,Emergency Assessment,583,21.2,35.4


## Directly-Follows edges (LEAD window function)

In [4]:
pd.read_sql('''SELECT activity, next_activity, COUNT(*) edge_count FROM (SELECT case_id, event_index, activity,
 LEAD(activity) OVER (PARTITION BY case_id ORDER BY event_index) next_activity FROM fact_event)
 WHERE next_activity IS NOT NULL GROUP BY activity, next_activity HAVING COUNT(*)>=3 ORDER BY edge_count DESC LIMIT 10''', eng)

,activity,next_activity,edge_count
0,Registration,Triage,1100
1,Discharge Planning,Discharge,979
2,Specialist Consultation,Lab Test,921
3,Physiotherapy,Discharge Planning,775
4,Diagnosis Confirmed,Medication Prescribed,725
5,Emergency Assessment,Lab Test,579
6,Radiology,Diagnosis Confirmed,579
7,Triage,Emergency Assessment,579
8,Lab Test,Diagnosis Confirmed,521
9,Medication Prescribed,Discharge,521
